# Problema #02 — Estimar el corte (cantidad de envíos) para Kaggle (Grupo A)

Con la función de ganancia (BAJA+2 suma 975000, cada envío de más resta 25000) conviene enviar a un cliente solo si su probabilidad de ser BAJA+2 supera **25000 / (975000 + 25000) = 2,5 %**. El corte óptimo es, entonces, la cantidad de clientes de 202109 cuya probabilidad de BAJA+2 pasa ese umbral.

Como no conocemos la verdad de 202109, se estima de tres maneras con las probabilidades ya guardadas (**no se entrena nada**):

1. **Teórico:** la probabilidad del modelo es la de ser BAJA+1 **o** BAJA+2. La de BAJA+2 es esa probabilidad por la proporción de BAJA+2 entre los positivos en validación. Corte = clientes de 202109 que pasan el 2,5 %.
2. **Teórico ajustado:** igual, pero corrigiendo por cuánto subestima el modelo en validación (positivos reales / positivos esperados).
3. **Isotónico:** en validación (202107) se ajusta una curva monótona entre la probabilidad y la tasa real de BAJA+2; se busca la probabilidad a partir de la cual la tasa es ≥ 2,5 % y se cuentan los clientes de 202109 que la superan.

Se informan además el **óptimo local** de la ganancia en 202107 (llevado a la escala de 202109) y el cociente entre los positivos esperados en 202109 y en 202107.

**Límites:** la estimación supone que el modelo se comporta en 202109 como en 202107, y ya vimos que el óptimo de Kaggle quedó por debajo del local. Los tres métodos dan un rango, no un número exacto. Para mezclas de métodos solo es válido con `PARAM$combinar <- "prob"`.

## 1. Inicialización y parámetros

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

require("data.table")

In [ ]:
PARAM <- list()

# donde estan las carpetas WF96xx
PARAM$carpeta_exp <- "/content/buckets/b1/exp"
PARAM$experimento_base <- 9600

# MISMO orden que metodos_DR del notebook 06 (define el numero de carpeta)
metodos_DR <- c(
  "ninguno", "deflacion", "dolar_blue", "dolar_oficial",
  "rank_simple", "rank_cero_fijo", "estandarizar",
  "ninguno_pond", "deflacion_pond",
  "deflacion_selectiva"
)

PARAM$semillas <- c(
  165463, 329969, 454253, 593899, 980249,
  100183, 250027, 400031, 550049, 700057
)

PARAM$carpeta_salida <- file.path(PARAM$carpeta_exp, paste0("BLEND_", PARAM$experimento_base))

# como se combinan los metodos dentro de una mezcla:
#  "prob" = promedio de las probabilidades | "rank" = promedio de los rankings (percentiles)
PARAM$combinar <- "prob"

# candidatos: nombre = c(metodos que lo forman). Un solo metodo = ese metodo solo.
#  (si falta alguno, el candidato se saltea y se avisa)
PARAM$candidatos <- list(
  ninguno        = "ninguno",
  estandarizar   = "estandarizar",
  dolar_oficial  = "dolar_oficial",
  rank_cero_fijo = "rank_cero_fijo",
  mezcla_ninguno_estandarizar = c("ninguno", "estandarizar"),
  mezcla_4 = c("ninguno", "estandarizar", "dolar_oficial", "rank_cero_fijo"),
  mezcla_6 = c("ninguno", "dolar_blue", "dolar_oficial", "rank_cero_fijo", "rank_simple", "estandarizar")
)

# cortes (cantidad de envios sobre los clientes de 202109) de los archivos de Kaggle a generar
PARAM$cortes_kaggle <- seq(1000, 2200, by = 100)

# cortes de la curva local (se llevan a la proporcion de 202107)
PARAM$cortes_local <- seq(600, 3000, by = 100)

# rango de cortes del enunciado (Grupo A) para el promedio de referencia
PARAM$rango_enunciado <- c(1800, 2400)

# ganancia: solo BAJA+2 suma (el costo -25000 es un SUPUESTO)
PARAM$ganancia$acierto <- 975000
PARAM$ganancia$costo <- -25000

PARAM$kaggle$competencia <- "labo-1-virtual-2026-analista-jr"

dir.create(PARAM$carpeta_salida, showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(PARAM$carpeta_salida, "kaggle"), showWarnings = FALSE)

stopifnot(PARAM$combinar %in% c("prob", "rank"))

## 2. Lectura de las probabilidades de cada método
Por método: la probabilidad de validación (promedio de las 10 semillas, 202107) y la del futuro (`prediccion.txt`, 202109).

In [ ]:
carpeta_metodo <- function(m) {
  file.path(PARAM$carpeta_exp, paste0("WF", PARAM$experimento_base + match(m, metodos_DR) - 1))
}

leer_metodo <- function(m) {
  carpeta <- carpeta_metodo(m)
  f_val <- file.path(carpeta, paste0("prediccion_val_", PARAM$semillas, ".txt"))
  f_fut <- file.path(carpeta, "prediccion.txt")
  f_cla <- file.path(carpeta, "validacion_clases.txt")
  if (!all(file.exists(c(f_val, f_fut, f_cla)))) return(NULL)

  val <- lapply(f_val, fread)
  ids <- val[[1]]$numero_de_cliente
  stopifnot(all(vapply(val, function(d) identical(d$numero_de_cliente, ids), logical(1))))

  list(
    val = data.table(numero_de_cliente = ids, prob = rowMeans(vapply(val, function(d) d$prob, numeric(length(ids))))),
    fut = fread(f_fut),
    clases = fread(f_cla)
  )
}

nombres_metodos <- unique(unlist(PARAM$candidatos))
stopifnot(all(nombres_metodos %in% metodos_DR))

datos <- lapply(setNames(nombres_metodos, nombres_metodos), leer_metodo)
faltan <- names(datos)[vapply(datos, is.null, logical(1))]
if (length(faltan) > 0) cat("metodos sin corrida completa (se ignoran los candidatos que los usan):", faltan, "\n")
datos <- datos[!vapply(datos, is.null, logical(1))]

# todas las corridas tienen que tener los mismos clientes (misma foto_mes) y las mismas clases
ref <- datos[[1]]
for (m in names(datos)) {
  stopifnot(
    setequal(datos[[m]]$val$numero_de_cliente, ref$val$numero_de_cliente),
    setequal(datos[[m]]$fut$numero_de_cliente, ref$fut$numero_de_cliente)
  )
}
clases <- ref$clases[match(ref$val$numero_de_cliente, numero_de_cliente), clase_ternaria]
cat("clientes validacion:", nrow(ref$val), "| BAJA+2:", sum(clases == "BAJA+2"),
  "| clientes futuro:", nrow(ref$fut), "\n")

## 3. Armado de los candidatos (un método o una mezcla)

In [ ]:
puntaje <- function(tb, ids) {
  p <- tb$prob[match(ids, tb$numero_de_cliente)]
  if (PARAM$combinar == "rank") p <- frank(p, ties.method = "average") / length(p)
  p
}

armar_candidato <- function(metodos) {
  list(
    val = rowMeans(vapply(metodos, function(m) puntaje(datos[[m]]$val, ref$val$numero_de_cliente),
      numeric(nrow(ref$val)))),
    fut = rowMeans(vapply(metodos, function(m) puntaje(datos[[m]]$fut, ref$fut$numero_de_cliente),
      numeric(nrow(ref$fut))))
  )
}

candidatos <- PARAM$candidatos[vapply(PARAM$candidatos, function(ms) all(ms %in% names(datos)), logical(1))]
stopifnot(length(candidatos) > 0)
cand <- lapply(candidatos, armar_candidato)
names(cand)

## 4. Estimación del corte por candidato

In [ ]:
stopifnot(PARAM$combinar == "prob")

positivos <- clases != "CONTINUA"
share_baja2 <- sum(clases == "BAJA+2") / sum(positivos) # BAJA+2 entre los positivos de validacion
umbral_tasa <- -PARAM$ganancia$costo / (PARAM$ganancia$acierto - PARAM$ganancia$costo) # 0.025
y2 <- as.integer(clases == "BAJA+2")
cat("BAJA+2 entre los positivos en validacion:", round(share_baja2, 3), "| umbral de tasa:", umbral_tasa, "\n")

estimar_corte <- function(x) {
  esperados_val <- sum(x$val)
  esperados_fut <- sum(x$fut)
  factor <- sum(positivos) / esperados_val # cuanto subestima el modelo en validacion

  # 1) teorico
  thr1 <- umbral_tasa / share_baja2
  # 2) teorico ajustado por la calibracion global
  thr2 <- umbral_tasa / (share_baja2 * factor)
  # 3) isotonico sobre la validacion
  o <- order(x$val)
  fit <- isoreg(x$val[o], y2[o])$yf
  idx <- which(fit >= umbral_tasa)
  thr3 <- if (length(idx) > 0) min(x$val[o][idx]) else Inf

  # optimo local (ganancia en 202107), llevado a la escala de 202109
  g <- ganancia_cortes(x$val, clases, cortes_val)
  corte_local <- PARAM$cortes_local[which.max(g)]

  data.table(
    esperados_val = round(esperados_val), positivos_reales_val = sum(positivos),
    esperados_fut = round(esperados_fut), cociente_fut_val = round(esperados_fut / esperados_val, 3),
    corte_teorico = sum(x$fut >= thr1),
    corte_teorico_ajustado = sum(x$fut >= thr2),
    corte_isotonico = sum(x$fut >= thr3),
    corte_optimo_local = corte_local
  )
}

In [ ]:
ganancia_cortes <- function(score, clases, cortes_val) {
  orden <- order(score, decreasing = TRUE)
  gan <- ifelse(clases[orden] == "BAJA+2", PARAM$ganancia$acierto, PARAM$ganancia$costo)
  cumsum(gan)[pmin(cortes_val, length(gan))]
}
escala <- nrow(ref$val) / nrow(ref$fut)
cortes_val <- pmax(1L, as.integer(round(PARAM$cortes_local * escala)))

est <- rbindlist(lapply(names(cand), function(n) cbind(candidato = n, estimar_corte(cand[[n]]))))
fwrite(est, file.path(PARAM$carpeta_salida, "estimacion_corte.csv"))
est

## 5. Tasa real de BAJA+2 por tramo del ranking (validación 202107)
Muestra dónde la tasa cruza el umbral de 2,5 %. Cada tramo tiene pocos clientes: es ruidoso.

In [ ]:
tramos <- rbindlist(lapply(names(cand), function(n) {
  r <- rank(-cand[[n]]$val, ties.method = "first")
  desde <- seq(1, 3000, by = 200)
  data.table(candidato = n, desde = desde, hasta = desde + 199,
    tasa_baja2_pct = vapply(desde, function(a) 100 * mean(y2[r >= a & r <= a + 199]), numeric(1)))
}))
fwrite(tramos, file.path(PARAM$carpeta_salida, "tasa_por_tramo.csv"))
dcast(tramos, desde + hasta ~ candidato, value.var = "tasa_baja2_pct")[, lapply(.SD, function(z) round(z, 1))]